This script scans each directory for nuSVR_Rerun_*.txt files generated during the nuSVR hyperparameter search. For every sample, it reads the RMSE-PredictedCounts metric, identifies the C and nu combination that yields the lowest RMSE, and copies that file as the final output (nuSVR_CountsRMSE_Random_<sample>.txt). This replicates the selection logic of the slower original script but for the problematic samples.

In [1]:
import os
import glob
import pandas as pd
import shutil

# find all directories matching the pattern
dir_pattern = "Decon-COO-Results_v3_Noise_phi*_nuSVR/"
directories = sorted(glob.glob(dir_pattern))

print(f"Found {len(directories)} directories.")

for input_dir in directories:
    print(f"\nProcessing directory: {input_dir}")

    pattern = os.path.join(input_dir, "nuSVR_Rerun_*.txt")
    files = glob.glob(pattern)

    if not files:
        print("  No rerun files found — skipping.")
        continue

    samples = {}

    for f in files:
        name = os.path.basename(f)
        parts = name.replace(".txt", "").split("_")

        # sample ID = everything between 3rd token and C/nu
        sample_id = "_".join(parts[3:-2])

        # C and nu always at the end
        C_val = parts[-2].replace("C", "")
        nu_val = parts[-1].replace("nu", "")

        df = pd.read_csv(f, sep="\t")
        rmse = float(df["RMSE-PredictedCounts"].iloc[0])

        # keep best RMSE per sample
        if sample_id not in samples or rmse < samples[sample_id]["rmse"]:
            samples[sample_id] = {
                "rmse": rmse,
                "file": f,
                "C": C_val,
                "nu": nu_val
            }

    # copy and rename winning files
    for sample_id, info in samples.items():
        best_file = info["file"]
        rmse = info["rmse"]
        C_val = info["C"]
        nu_val = info["nu"]

        # output format you requested
        new_name = f"nuSVR_CountsRMSE_Random_{sample_id}.txt"
        new_path = os.path.join(input_dir, new_name)

        shutil.copy(best_file, new_path)

        print(f"  Sample: {sample_id}")
        print(f"    → Best RMSE: {rmse:.6f}")
        print(f"    → Best C:  {C_val}")
        print(f"    → Best nu: {nu_val}")
        print(f"    → Saved: {new_path}")


Found 9 directories.

Processing directory: Decon-COO-Results_v3_Noise_phi100_nuSVR/
  Sample: TSP21_random_rep1_seed10_10_200
    → Best RMSE: 0.243041
    → Best C:  1
    → Best nu: 0.75
    → Saved: Decon-COO-Results_v3_Noise_phi100_nuSVR/nuSVR_CountsRMSE_Random_TSP21_random_rep1_seed10_10_200.txt
  Sample: TSP27_random_rep5_seed10_40_800
    → Best RMSE: 0.181668
    → Best C:  0.5
    → Best nu: 0.75
    → Saved: Decon-COO-Results_v3_Noise_phi100_nuSVR/nuSVR_CountsRMSE_Random_TSP27_random_rep5_seed10_40_800.txt
  Sample: TSP21_random_rep1_seed9_20_400
    → Best RMSE: 0.870151
    → Best C:  10
    → Best nu: 0.25
    → Saved: Decon-COO-Results_v3_Noise_phi100_nuSVR/nuSVR_CountsRMSE_Random_TSP21_random_rep1_seed9_20_400.txt
  Sample: TSP27_random_rep2_seed5_10_200
    → Best RMSE: 0.850509
    → Best C:  0.5
    → Best nu: 0.25
    → Saved: Decon-COO-Results_v3_Noise_phi100_nuSVR/nuSVR_CountsRMSE_Random_TSP27_random_rep2_seed5_10_200.txt
  Sample: TSP21_random_rep4_seed8_10_200
 